# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /home/swathi/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/swathi/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [2]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [4]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Use-Case Data!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [5]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [6]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

/home/swathi/AIProjects/AIE2/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
/home/swathi/AIProjects/AIE2/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/lang/arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
/home/swathi/AIProjects/AIE2/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/lang/persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)


Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [7]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [8]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 64, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [9]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/38 [00:00<?, ?it/s]

Property 'summary' already exists in node '18f0e7'. Skipping!
Property 'summary' already exists in node '6a04eb'. Skipping!
Property 'summary' already exists in node '1cbce2'. Skipping!
Property 'summary' already exists in node '6cc7cc'. Skipping!
Property 'summary' already exists in node '550307'. Skipping!
Property 'summary' already exists in node 'fb7833'. Skipping!
Property 'summary' already exists in node '6d2891'. Skipping!
Property 'summary' already exists in node 'a02316'. Skipping!
Property 'summary' already exists in node '21a107'. Skipping!
Property 'summary' already exists in node '972f0e'. Skipping!
Property 'summary' already exists in node 'cce53f'. Skipping!
Property 'summary' already exists in node '51cbee'. Skipping!
Property 'summary' already exists in node 'a323b9'. Skipping!
Property 'summary' already exists in node '4228c4'. Skipping!
Property 'summary' already exists in node '07c148'. Skipping!
Property 'summary' already exists in node '9db1d6'. Skipping!
Property

Applying CustomNodeFilter:   0%|          | 0/8 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/48 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node '6cc7cc'. Skipping!
Property 'summary_embedding' already exists in node '6a04eb'. Skipping!
Property 'summary_embedding' already exists in node '1cbce2'. Skipping!
Property 'summary_embedding' already exists in node '550307'. Skipping!
Property 'summary_embedding' already exists in node 'fb7833'. Skipping!
Property 'summary_embedding' already exists in node '18f0e7'. Skipping!
Property 'summary_embedding' already exists in node 'a02316'. Skipping!
Property 'summary_embedding' already exists in node '21a107'. Skipping!
Property 'summary_embedding' already exists in node '6d2891'. Skipping!
Property 'summary_embedding' already exists in node 'cce53f'. Skipping!
Property 'summary_embedding' already exists in node 'a323b9'. Skipping!
Property 'summary_embedding' already exists in node '972f0e'. Skipping!
Property 'summary_embedding' already exists in node '4228c4'. Skipping!
Property 'summary_embedding' already exists in node '07c148'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 86, relationships: 711)

We can save and load our knowledge graphs as follows.

In [10]:
kg.save("usecase_data_kg.json")
usecase_data_kg = KnowledgeGraph.load("usecase_data_kg.json")
usecase_data_kg

KnowledgeGraph(nodes: 86, relationships: 711)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [14]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=usecase_data_kg)





However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [15]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

SingleHopSpecificQuerySynthesizer (50% weight):
Generates questions that require one step to answer
Questions are specific and direct
Example: "What is the capital of France?" (direct fact retrieval)

MultiHopAbstractQuerySynthesizer (25% weight):
Generates questions requiring multiple steps to answer
Questions are abstract and require reasoning
Example: "How do economic policies affect social mobility?" (requires connecting multiple concepts)

MultiHopSpecificQuerySynthesizer (25% weight):
Generates questions requiring multiple steps to answer
Questions are specific but complex
Example: "What are the specific steps in the machine learning pipeline mentioned in the document?" (specific but multi-step)
Each synthesizer uses personas and scenarios to create realistic questions that someone would ask about the data.


In [16]:

testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,"Eloundou et al., 2025 data privacy?",[Introduction ChatGPT launched in November 202...,"The context mentions Eloundou et al., 2025 in ...",single_hop_specifc_query_synthesizer
1,What is the significance of June 2024?,[Table 1: ChatGPT daily message counts (millio...,The context reports on data ending on the 26th...,single_hop_specifc_query_synthesizer
2,What is Appendix D about?,[Variation by Occupation Figure 23 presents va...,Appendix D contains a full report of GWA count...,single_hop_specifc_query_synthesizer
3,WhaT is the meaning of Seeking Information in ...,[Conclusion This paper studies the rapid growt...,Seeking Information is one of the three most c...,single_hop_specifc_query_synthesizer
4,"How does message classification into Asking, D...",[<1-hop>\n\nConclusion This paper studies the ...,The context explains that messages sent to Cha...,multi_hop_abstract_query_synthesizer
5,Considering the classification of messages int...,[<1-hop>\n\nTable 1: ChatGPT daily message cou...,The significant rise in non-work-related messa...,multi_hop_abstract_query_synthesizer
6,Based on the data showing the growth of non-wo...,[<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (%...,The context indicates that non-work messages h...,multi_hop_abstract_query_synthesizer
7,Based on the data showing that ChatGPT usage i...,[<1-hop>\n\nConclusion This paper studies the ...,The rapid growth of ChatGPT's non-work-related...,multi_hop_specific_query_synthesizer
8,US like how much ChatGPT data is used in US an...,[<1-hop>\n\nConclusion This paper studies the ...,"The context indicates that in the US, ChatGPT ...",multi_hop_specific_query_synthesizer
9,How does the rapid growth of ChatGPT to over 7...,[<1-hop>\n\nConclusion This paper studies the ...,The first context segment states that by July ...,multi_hop_specific_query_synthesizer


Finally, we can use our `TestSetGenerator` to generate our testset!

### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [17]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/38 [00:00<?, ?it/s]

Property 'summary' already exists in node 'e455f3'. Skipping!
Property 'summary' already exists in node '708c00'. Skipping!
Property 'summary' already exists in node 'c36ed7'. Skipping!
Property 'summary' already exists in node 'f23da7'. Skipping!
Property 'summary' already exists in node 'cc927b'. Skipping!
Property 'summary' already exists in node 'a761d5'. Skipping!
Property 'summary' already exists in node '8d776b'. Skipping!
Property 'summary' already exists in node '7f98f7'. Skipping!
Property 'summary' already exists in node '30c442'. Skipping!
Property 'summary' already exists in node '9f659f'. Skipping!
Property 'summary' already exists in node '7fe50e'. Skipping!
Property 'summary' already exists in node '626b83'. Skipping!
Property 'summary' already exists in node 'a2342a'. Skipping!
Property 'summary' already exists in node 'd17509'. Skipping!
Property 'summary' already exists in node '2d785e'. Skipping!
Property 'summary' already exists in node '747950'. Skipping!
Property

...

Property 'summary_embedding' already exists in node '626b83'. Skipping!
Property 'summary_embedding' already exists in node '708c00'. Skipping!
Property 'summary_embedding' already exists in node 'e455f3'. Skipping!
Property 'summary_embedding' already exists in node '7f98f7'. Skipping!
Property 'summary_embedding' already exists in node 'f23da7'. Skipping!
Property 'summary_embedding' already exists in node 'c36ed7'. Skipping!
Property 'summary_embedding' already exists in node '8d776b'. Skipping!
Property 'summary_embedding' already exists in node 'cc927b'. Skipping!
Property 'summary_embedding' already exists in node '9f659f'. Skipping!
Property 'summary_embedding' already exists in node '7fe50e'. Skipping!
Property 'summary_embedding' already exists in node 'a761d5'. Skipping!
Property 'summary_embedding' already exists in node '30c442'. Skipping!
Property 'summary_embedding' already exists in node 'a2342a'. Skipping!
Property 'summary_embedding' already exists in node '2d785e'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [18]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,"What does Tomlinson et al., 2025 contribute to...",[Introduction ChatGPT launched in November 202...,"Tomlinson et al., 2025 report statistics on ch...",single_hop_specifc_query_synthesizer
1,What is the significance of June 2024 in the c...,[Table 1: ChatGPT daily message counts (millio...,June 2024 is the end date for the 7-day averag...,single_hop_specifc_query_synthesizer
2,What is the significance of Appendx D in the r...,[Variation by Occupation Figure 23 presents va...,Appendix D contains a full report of GWA count...,single_hop_specifc_query_synthesizer
3,What about July 2025 is important for AI resea...,[Conclusion This paper studies the rapid growt...,"By July 2025, ChatGPT had been used weekly by ...",single_hop_specifc_query_synthesizer
4,how the usage of chatgpt messages changed betw...,[<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (%...,"In June 2024, the total messages were 451 mill...",multi_hop_abstract_query_synthesizer
5,how much messages chatgpt users send in jun 20...,[<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (%...,"In June 2024, total messages were 451 million,...",multi_hop_abstract_query_synthesizer
6,Based on the data showing that non-work messag...,[<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (%...,"The data indicates that by June 2025, non-work...",multi_hop_abstract_query_synthesizer
7,Based on the observed changes in user behavior...,[<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (%...,The data indicates that from June 2024 to June...,multi_hop_abstract_query_synthesizer
8,Whatt hapen in July 2025 with ChatGPT and how ...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"By July 2025, ChatGPT had been used weekly by ...",multi_hop_specific_query_synthesizer
9,what happened in july 2025 with chatgpt and ho...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"In july 2025, chatgpt was used weekly by more ...",multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [19]:
from langsmith import Client

client = Client()

dataset_name = "Use Case Synthetic Data - AIE8"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic Data for Use Cases"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [20]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [21]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [22]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [23]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [24]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG"
)

In [25]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [26]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

As is usual: We'll be using `gpt-4.1-mini` for our RAG!

In [27]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [28]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [29]:
rag_chain.invoke({"question" : "What are people doing with AI these days?"})

'Based on the provided context, people are using AI, particularly generative AI like ChatGPT, in a variety of ways both at work and outside of work. They use AI to perform workplace tasks by either augmenting or automating human labor. AI is used for producing writing, software code, spreadsheets, and other digital products, distinguishing it from traditional technologies like web search engines. People engage with AI with different intents classified as Asking (seeking information or advice), Doing (producing output), and Expressing (self-expression). Some occupational implications include AI serving as co-workers producing output or as co-pilots improving human problem-solving productivity. Additionally, uses include relationships and personal reflection, games and role play, therapy/companionship, and data analysis, though with varying prevalence.\n\nIn summary, people are using AI for workplace productivity, creative and digital content generation, seeking information and advice, p

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [30]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [31]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

dopeness_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "dopeness": "Is this response dope, lit, cool, or is it just a generic response?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

- `qa_evaluator`:

    Evaluates the accuracy and correctness of answers. Uses LangChain's built-in QA evaluator. measures Whether the generated answer correctly addresses the question based on the provided context.

- `labeled_helpfulness_evaluator`:
Evaluates the helpfulness of responses. Uses labeled criteria evaluation with reference answers , measures Whether the submission is helpful to the user, taking into account the correct reference answer, Compares against ground truth reference answers.

- `dopeness_evaluator`:
Evaluates the quality and engagement of responses, Uses criteria-based evaluation, measures Whether the response is "dope, lit, cool" or just generic. Focus on Measuring creativity, engagement, and non-generic responses

## LangSmith Evaluation

In [32]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'abandoned-memory-45' at:
https://smith.langchain.com/o/c8a2b29e-9189-4b26-aeda-9228a3fd27b1/datasets/60600086-8d6d-45d3-aa3b-d56d8f037e97/compare?selectedSessions=e2bbd16a-7960-44ff-8d27-15636a162c07




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,"Based on the findings of Handa et al. (2025), ...",I don't know.,None,"According to Handa et al. (2025), the data sho...",0,0,0,1.903444,d4a05134-76b0-46a5-9158-3a4e6b75368d,6d4d05d5-3172-44fe-b0f2-eac80bcb76d6
1,"How does the growth in ChatGPT usage, as discu...","Based on the provided context, the growth in C...",None,"According to the context, Handa et al. (2025) ...",0,1,0,2.318528,b9c32a70-e888-4a2b-a7f6-fc5c1709c2e9,7048ccc8-4576-4a7f-bb66-2a06adceed96
2,what happened in july 2025 with chatgpt and ho...,"In July 2025, ChatGPT had more than 700 millio...",None,"In july 2025, chatgpt was used weekly by more ...",1,1,0,1.337769,edd3c43f-b14d-40e6-aeaf-ece42846f352,9716aae4-fdb9-4398-a6c5-0a1298d41fa9
3,Whatt hapen in July 2025 with ChatGPT and how ...,"In July 2025, ChatGPT had been used weekly by ...",None,"By July 2025, ChatGPT had been used weekly by ...",1,1,0,3.144778,b738d4d6-c6cb-4005-b4d9-37523844e241,1c5053c5-9252-404e-9332-bdd919cf4508
4,Based on the observed changes in user behavior...,"Based on the provided context, between June 20...",None,The data indicates that from June 2024 to June...,1,1,0,10.573590,8c1bfed7-0a7e-4d67-adf6-3eb16f2d14ed,61c0e910-19b7-4483-bcd7-ed76f56108af
5,Based on the data showing that non-work messag...,The significant rise in non-work message volum...,None,"The data indicates that by June 2025, non-work...",1,1,0,3.127825,0e3447d1-203f-4172-9901-0e9fea5d5d4e,b4d084ea-988d-4a7b-9fbd-59a4127402b7
6,how much messages chatgpt users send in jun 20...,"Between July 2024 and July 2025, the number of...",None,"In June 2024, total messages were 451 million,...",1,0,0,2.655313,14f151cb-f871-417d-8c67-b8c9613d5dd8,188c61bd-d2c9-488c-9971-11fdf1d0cdf2
7,how the usage of chatgpt messages changed betw...,"Between June 2024 and June 2025, the usage of ...",None,"In June 2024, the total messages were 451 mill...",1,1,0,10.095470,db751493-d1e7-4666-afd2-8b1461cd4290,9883b103-cc94-4a18-a891-7fc5d23b20e7
8,What about July 2025 is important for AI resea...,"In July 2025, OpenAI reported that ChatGPT use...",None,"By July 2025, ChatGPT had been used weekly by ...",1,0,0,1.151332,1f996d0c-8d2c-48b0-9b5b-647cc910db60,5d1f114a-e394-479f-98e9-cf9ce4375f13
9,What is the significance of Appendx D in the r...,Appendix D in the report provides occupational...,None,Appendix D contains a full report of GWA count...,1,1,0,2.134618,7b38e6b5-8889-4714-b1c5-426fc087085b,8f66a70f-df3b-4003-b303-46b3892c8f76


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [33]:
DOPENESS_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Make your answer rad, ensure high levels of dopeness. Do not be generic, or give generic responses.

Context: {context}
Question: {question}
"""

dopeness_rag_prompt = ChatPromptTemplate.from_template(DOPENESS_RAG_PROMPT)

In [34]:
rag_documents = docs

In [35]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

Larger chuck size gives better context, more complete information per chunk. It improves coherence, better for multi-hop reasoning questions and with reduced fragmentation, less likely to split related concepts. The synthetic test data includes multi-hop questions that require connecting information across different parts of documents. Larger chunks provide more context for the LLM to make these connections, leading to better performance on complex queries.

In [36]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

Upgrading from text-embedding-3-small to text-embedding-3-large improves RAG performance through several mechanisms:

Technical Differences:
text-embedding-3-small: 1536 dimensions, faster and cheaper
text-embedding-3-large: 3072 dimensions, more sophisticated representations

Performance Improvements:
Better Semantic Understanding:
Larger model captures more nuanced relationships between concepts
Better at understanding context and meaning beyond simple keyword matching

Improved Retrieval Accuracy:
More precise similarity calculations
Better handling of synonyms, paraphrases, and conceptual relationships
Reduced false positives in retrieval

Enhanced Multi-hop Reasoning:
Better at connecting related concepts across different documents
Improved performance on complex queries that require reasoning

Domain Adaptation:
Better performance on domain-specific terminology
More robust handling of technical concepts

Trade-offs:
Cost: Higher computational and API costs
Latency: Slightly slower embedding generation
Memory: Larger vector dimensions require more storage

The synthetic test data includes abstract and multi-hop questions that require sophisticated semantic understanding. The larger embedding model provides better semantic representations, leading to more accurate retrieval and better overall RAG performance.

In [37]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG Docs"
)

In [38]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [39]:
dopeness_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dopeness_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [40]:
dopeness_rag_chain.invoke({"question" : "How are people using AI to make money?"})

'Alright, buckle up — here’s the scoop on how humans are turning AI into pure $$$ based on the slick insights from the context:\n\nPeople ain’t just using AI to do grunt tasks; they’re treating ChatGPT like their personal advisor and research assistant. It’s like having a turbocharged brainiac in your corner helping you make smarter, faster decisions. This is especially clutch in knowledge-heavy jobs where the quality of your choices can make or break your paycheck.\n\nCollis and Brynjolfsson (2025) drop a bomb: US users value generative AI so much that folks would need to be paid a cool $98 just to *not* use it for a month. That hints at a massive $97 billion consumer surplus annually just from the edge AI gives people.\n\nSo, the real moneymaker? Using AI to *augment* your hustle — leveraging it for top-tier decision support, killer research, and advisory roles, which cranks up productivity and ultimately cash flow. It’s not just automation; it’s AI as your strategic wingman, helping

Finally, we can evaluate the new chain on the same test set!

In [41]:
evaluate(
    dopeness_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "dopeness_rag_chain"},
)

View the evaluation results for experiment: 'impressionable-protest-3' at:
https://smith.langchain.com/o/c8a2b29e-9189-4b26-aeda-9228a3fd27b1/datasets/60600086-8d6d-45d3-aa3b-d56d8f037e97/compare?selectedSessions=175a5892-fbf6-481a-8de0-fe6a3c5d0a58




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,"Based on the findings of Handa et al. (2025), ...","Yo, strap in—this is the dopest deep dive into...",None,"According to Handa et al. (2025), the data sho...",1,1,1,4.603229,d4a05134-76b0-46a5-9158-3a4e6b75368d,227d6069-94d8-4388-b304-aa0c8748f052
1,"How does the growth in ChatGPT usage, as discu...","Yo, let’s crank this up to eleven — here’s the...",None,"According to the context, Handa et al. (2025) ...",1,0,1,6.254625,b9c32a70-e888-4a2b-a7f6-fc5c1709c2e9,fb16adad-e7ab-40b7-ab4c-a8fd53449c04
2,what happened in july 2025 with chatgpt and ho...,"Yo, July 2025 was a straight-up fireworks show...",None,"In july 2025, chatgpt was used weekly by more ...",1,1,1,1.189009,edd3c43f-b14d-40e6-aeaf-ece42846f352,62f950b3-7729-4ac9-9e5e-ed9eda05f463
3,Whatt hapen in July 2025 with ChatGPT and how ...,"Yo, check this out—by July 2025, ChatGPT had b...",None,"By July 2025, ChatGPT had been used weekly by ...",1,1,1,4.445169,b738d4d6-c6cb-4005-b4d9-37523844e241,440dfcb0-3f5b-4f0c-9a63-d43ed5271a76
4,Based on the observed changes in user behavior...,"Oh, buckle up, because we’re diving deep into ...",None,The data indicates that from June 2024 to June...,1,1,1,10.368184,8c1bfed7-0a7e-4d67-adf6-3eb16f2d14ed,b52fcc3f-57fd-4a62-b6d8-acfc20769da9
5,Based on the data showing that non-work messag...,"Yo, let’s get extra fly with this: The surge i...",None,"The data indicates that by June 2025, non-work...",1,1,1,4.086991,0e3447d1-203f-4172-9901-0e9fea5d5d4e,aa223dec-8ab1-44a3-b313-b473c70407f0
6,how much messages chatgpt users send in jun 20...,"Yo, let’s dive into the slick stats from the C...",None,"In June 2024, total messages were 451 million,...",1,1,1,8.256718,14f151cb-f871-417d-8c67-b8c9613d5dd8,13aeb7dd-553b-46b9-b1de-6d45564f6ce7
7,how the usage of chatgpt messages changed betw...,"Alright, strap in for this wild ride through C...",None,"In June 2024, the total messages were 451 mill...",1,1,1,7.744661,db751493-d1e7-4666-afd2-8b1461cd4290,388a6f66-579d-408b-877c-d5e2c1a78824
8,What about July 2025 is important for AI resea...,"Yo, July 2025 wasn’t just any month—it was a s...",None,"By July 2025, ChatGPT had been used weekly by ...",1,0,1,3.575966,1f996d0c-8d2c-48b0-9b5b-647cc910db60,a854b1a2-ea1d-4a81-b990-79fb80e16db9
9,What is the significance of Appendx D in the r...,"Yo, Appendix D in the report is straight-up th...",None,Appendix D contains a full report of GWA count...,1,1,1,5.105970,7b38e6b5-8889-4714-b1c5-426fc087085b,4605383f-604d-438d-9118-23ecbd98f6ec


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.

The notebook evaluates two RAG chains on the same synthetic test dataset:
Default Chain: Basic RAG with smaller chunks and smaller embedding model
Dopeness Chain: Enhanced RAG with larger chunks, larger embedding model, and improved prompting

Expected Performance Changes:

QA Evaluator (Accuracy):
Likely Improvement: Better retrieval accuracy from larger embedding model
Reason: More precise semantic matching leads to better context retrieval

Labeled Helpfulness Evaluator:
Likely Improvement: Better context understanding improves answer quality
Reason: Larger chunks provide more complete context for generating helpful answers

Dopeness Evaluator:
Significant Improvement: Explicit prompting for "dopeness" should dramatically improve scores
Reason: The prompt specifically instructs the model to avoid generic responses and be more engaging

Multiple evaluators provide different perspectives on system performance. The evaluation results are automatically tracked in LangSmith, allowing for:
Performance comparison between different chain versions
Detailed analysis of individual question-answer pairs
Historical tracking of improvements over time
